# Lab 13: YOLO Training

**Module 13** | Read `notes/13-yolo-map.pdf` before running this notebook.

Fine-tune YOLOv8n on COCO128 and read off mAP@0.5 from the validation metrics.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## YOLOv8 training on COCO128

YOLO (You Only Look Once) predicts bounding boxes in a single forward pass. Ultralytics YOLOv8 wraps training, validation, and checkpointing in a high-level API.

COCO128 is a tiny 128-image subset of COCO bundled for smoke tests. On first run Ultralytics downloads both the `yolov8n.pt` weights and the COCO128 images automatically, expect a short delay and network use.


### Training configuration

Before calling `train()`, print the hyperparameters you will use. The nano variant (`yolov8n`) keeps runtime short while still exercising the full train/val/inference pipeline.


In [ ]:
from ultralytics import YOLO

TRAIN_CONFIG = {
    "model": "yolov8n.pt",
    "data": "coco128.yaml",
    "epochs": 10,
    "imgsz": 640,
    "batch": 16,
    "device": str(device),
}

print("YOLOv8 training configuration:")
for key, value in TRAIN_CONFIG.items():
    print(f"  {key}: {value}")
print()

model = YOLO(TRAIN_CONFIG["model"])
train_results = model.train(
    data=TRAIN_CONFIG["data"],
    epochs=TRAIN_CONFIG["epochs"],
    imgsz=TRAIN_CONFIG["imgsz"],
    batch=TRAIN_CONFIG["batch"],
    device=TRAIN_CONFIG["device"],
    verbose=True,
)


### Training results

After training completes, inspect the run directory and key metrics returned by the trainer. `save_dir` points to weights, plots, and logs under `runs/detect/`.


In [ ]:
print("Training complete.")
print()
print("Run artifacts:")
print(f"  Save directory: {train_results.save_dir}")
if hasattr(train_results, "best_fitness") and train_results.best_fitness is not None:
    print(f"  Best fitness:   {train_results.best_fitness:.4f}")
if hasattr(train_results, "fitness") and train_results.fitness is not None:
    print(f"  Final fitness:  {train_results.fitness:.4f}")

results_dict = train_results.results_dict if hasattr(train_results, "results_dict") else {}
if results_dict:
    print()
    print("Key metrics from training results:")
    for key in sorted(results_dict.keys()):
        val = results_dict[key]
        if isinstance(val, float):
            print(f"  {key}: {val:.4f}")
        else:
            print(f"  {key}: {val}")


### Validation metrics (detailed)

`model.val()` re-runs the validation split and returns box metrics. mAP@0.5 (IoU threshold 0.5) is the standard single-threshold detection score. mAP@0.5:0.95 averages across IoU thresholds from 0.5 to 0.95 and is the primary COCO benchmark.


In [ ]:
metrics = model.val()

print("Validation metrics (detailed):")
print("=" * 50)
print(f"  mAP@0.5:      {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"  Precision:    {metrics.box.mp:.4f}")
print(f"  Recall:       {metrics.box.mr:.4f}")
print()

if hasattr(metrics.box, "maps") and metrics.box.maps is not None:
    per_class = metrics.box.maps
    print(f"  Per-class mAP@0.5:0.95 available for {len(per_class)} classes")
    top_indices = sorted(range(len(per_class)), key=lambda i: per_class[i], reverse=True)[:5]
    print("  Top 5 classes by mAP@0.5:0.95:")
    names = metrics.names if hasattr(metrics, "names") else {}
    for idx in top_indices:
        name = names.get(idx, f"class_{idx}")
        print(f"    {name}: {per_class[idx]:.4f}")

print("=" * 50)


### Inference on a sample image

Run the fine-tuned weights on a local JPG and print each predicted box with class name, confidence, and coordinates. This confirms the end-to-end pipeline from training through deployment-style inference.


In [ ]:
from pathlib import Path

ROOT = Path("..").resolve()
sample_dir = ROOT / "datasets" / "sample_images"
sample = next(iter(sorted(sample_dir.glob("*.jpg"))), None)

if sample:
    preds = model.predict(source=str(sample), imgsz=640, conf=0.25, verbose=False)
    result = preds[0]
    boxes = result.boxes

    print(f"Inference on: {sample.name}")
    print(f"Image size: {result.orig_shape[1]}x{result.orig_shape[0]} (W x H)")
    print(f"Detections (conf >= 0.25): {len(boxes)}")
    print("-" * 70)

    names = result.names
    for i, box in enumerate(boxes):
        xyxy = box.xyxy[0].tolist()
        conf = box.conf[0].item()
        cls_id = int(box.cls[0].item())
        cls_name = names[cls_id]
        print(
            f"  [{i + 1}] {cls_name:20s}  conf={conf:.3f}  "
            f"box=[{xyxy[0]:6.1f}, {xyxy[1]:6.1f}, {xyxy[2]:6.1f}, {xyxy[3]:6.1f}]"
        )

    print("-" * 70)
    result.show()
else:
    print("No sample JPG found in datasets/sample_images/.")
    print("Training and validation still completed successfully.")


### Evaluation

A successful COCO128 smoke test shows finite mAP values, a populated `save_dir`, and at least some detections on the sample image. Absolute mAP will be modest after only 10 epochs on 128 images; the goal is to verify the Ultralytics workflow, not to match production COCO leaderboard scores.


In [ ]:
print("YOLOv8 evaluation summary:")
print(f"  Epochs trained: {TRAIN_CONFIG['epochs']}")
print(f"  Image size:     {TRAIN_CONFIG['imgsz']}")
print(f"  mAP@0.5:        {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95:   {metrics.box.map:.4f}")
print(f"  Weights saved:  {train_results.save_dir}")
